# iMAT: Iteration, Sampling, and Divergence

## Setup

Import some useful python packages to get started

In [ ]:
# Import metworkpy itself
import metworkpy

# For handling the models
import cobra

# For data handling
import pandas as pd

# For random number generation, and arrays
import numpy as np

# Setup the numpy rng we will use
rng = np.random.default_rng(2735982735987)

Configure COBRApy to use the hybrid solver, this requires that OSQP and HiGHS are installed, but both are open source and freely available and can be installed with the 'hybrid' extra when installing MetworkPy: `pip install metworkpy[hybrid]`.  

We need to use this solver instead of the GLPK solver in order to handle the Mixed-Integer Linear Programs (MILP) required by iMAT based appraoches.

In [ ]:
cobra.Configuration().solver = "hybrid"

## Get an Example Cobra Model

For this tutorial we will use the Ecoli Core model from BiGG, since it is small enough that it should be relatively fast, but still large enough that there can be multiple iMAT solutions that can iterated through using the iMAT iter approaches. 

In [ ]:
# Name of the model is textbook
model = cobra.io.load_model("textbook")

In [ ]:
model

## Gene Weights

For iMAT, and its various methods we will need to have gene weights. In order to evaluate the divergence between models later, we will generate two sets of random weights.   

To input gene weight data into MetworkPy, we need a pandas Series whose index is the gene ids from the Model (any genes not in the model will be ignored). In order to get this from gene expression data you can use `metworkpy.utils.expr_to_imat_gene_weights` which uses the quantiles of the gene expression data to determine gene weights. You can also use a variety of other approaches to generate gene weights, such as differential gene expression.  

In this tutorial, we will be working with weights that only take values -1, 0, and 1 but the method as implemented in MetworkPy is more general than this, and you can assign essentially any floating point number as a gene weight (assigning NaN and Infs will not work). Negative values are treated as low expression; positive values are treated as high expression; and 0 is treated as intermediate and/or indeterminate. The weights are used as weights in the objective function, so a very negative value will highly weight having a reaction being turned off, while a small positive will lightly weight having a reaciton being turned on. 

In [ ]:
gene_weights1 = pd.Series(
    rng.choice(
        [-1, 0, 1], size=len(model.genes), replace=True, p=[0.1, 0.8, 0.1]
    ),
    index=model.genes.list_attr("id"),
)
# Get another set of weights for comparison
gene_weights2 = pd.Series(
    rng.choice(
        [-1, 0, 1], size=len(model.genes), replace=True, p=[0.1, 0.8, 0.1]
    ),
    index=model.genes.list_attr("id"),
)

## Reaction Weights

In order to actually apply the iMAT method to the model, we need to convert these gene weights into expression weights. This is done by evaluating the Gene-Protein-Reaction rules of the metabolic model using the gene weights. These rules are boolean rules which combine the gene weights with AND and OR expressions. AND expressions mean that both genes are required for a reaction to function (as in the case of multi-enzyme complexes), while OR expressions indicate that multiple alternate could allow for a specific reaction (as in the case of isozymes). These expressions can be nested, for example a reaction could require either a single gene, or a multi-enzyme complex in which case the rule could be 'GENE1 OR (GENE2 AND GENE3)'. 

In order to apply these GPR's to our gene weights, the boolean logic needs to be extended in order to handle more than just true and false. This can be done by using a maxmimum to replace OR, and a minimum to replace AND. If a reaction requires two genes to function (AND), then the minimum expression level of those two genes represents the that reaction's functionality. Similarly, if a reaction can make use of either of two proteins, than the maximum expression level of those two genes represents that reaction's functionality.   

MetworkPy includes a function for performing this for all the genes in the model:

In [ ]:
reaction_weights1 = metworkpy.gpr.gene_to_rxn_weights(model, gene_weights1)
reaction_weights2 = metworkpy.gpr.gene_to_rxn_weights(model, gene_weights2)

By default this uses the mininum and maximum as decribed above, but it can actually use any method of combining two numbers into a single output value. The alternative approach can be specified by passing a dict of "AND", and "OR" to the desired functions (which must take at least two values and return a single value), as the `fn_dict` argument of the `metworkpy.gpr.gene_to_rxn_weights` function.

## Running iMAT

iMAT describes a mixed-integer linear program which attempts to find a matching between reaction flux and gene expression (as represented by the reaction weights we have generated). The problem as described in (Shlomi et al., 2008) is:
$max_{v,y^+,y^-}(\sum_{i\in R_H} (y_i^+ + y_i^-) + \sum_{i \in R_L}y_i^+) $

$s.t.$

$ S \cdot v =0 $                                                  (1)

$ v_{min} \leq v \leq v_{max}$                                    (2)

$v_i+y_i^+(v_{min,i}+\epsilon) \geq v_{min,i}, i\in R_H$          (3)

$v_i +y_i^-(v_{max,i}+\epsilon) \leq v_{max,i}, i\in R_H$         (4)

$v_{min,i}(1-y_i^+)\leq v_i \leq v_{max,i} (1-y_i^+),i\in R_L$    (5)

$v \in R^m$

$y_i^+, y_i^- \in [0,1]$


For high expression reactions the $y_i^+$ represents active (flux greater than $\epsilon$ in the forward direction, $y_i^-$ represents active in the reverse direction. For low expression reactions, the $y_i^+$ represents an inactive reaction (flux equal to 0). Thus the objective is to maximize the sum of high expression reactions which are active (either forward or reverse) plus the sum of low expression reactions with are inactive. MetworkPy makes use of a later formulation that instead allows for the low expression reactions to have a flux below a threshold, this modified equation (5) to be 

$ v_{min, i}(1-y_i^+) - y_i^+(threshold)\leq v_i \leq v_{max,i} (1-y_i^+) + y_i^+(threshold),i\in R_L$.

MetworkPy includes a function for formulating and running this problem:

In [ ]:
metworkpy.imat.imat(
    model, rxn_weights=reaction_weights1, epsilon=1.0, threshold=1e-2
)

In [ ]:
# And for the second set of reaction weights
metworkpy.imat.imat(
    model, rxn_weights=reaction_weights2, epsilon=1.0, threshold=1e-2
)

## Generating a Model From iMAT

There are several approaches for actually generating a model from iMAT, the simplest method is just to create a model with the iMAT objective added as a constraint to the model. This approach is called 'imat_restrictions' in MetworkPy. This works, but it then requires solving a MILP to work with the Model which is computationally expensive, and it won't work with the sampling methods of COBRApy (which will be discussed more later, and MetworkPy includes a sampling method which will work with this model type though it is much slower).   

An alternate approach is just to constrain the reactions which were found to be inactive to always have a flux below threshold, called 'subset' in MetworkPy. Additional constraints can be added to restrict the active reactions to be active, that is carry a flux above $\epsilon$, which is called 'simple' in MetworkPy. The weakness of these approaches is that they make use of a single iMAT solution, and while the objective value is unique, the fluxes that generate it are not unique. This means there can be a variety of different sets of reactions which are considered active/inactive that all lead to the same optimum. In order to address this, MetworkPy includes two additional approaches. The first, called 'milp' is based on the approach in (Shlomi et al., 2008), which involves sequentially forcing reactions to be active/inactive and evaluating which of these is more consistant with the gene expression state. The second, called 'fva' instead applies flux-variability analysis to the 'imat_restrictions' model to identify the feasible flux bounds of each reaction which still maximize the iMAT objective, and these bounds are used to constraint the model.    

Both the 'milp' and 'fva' approaches can result in infeasible models, since they find the bounds for each reaction separately. Adjusting the parameters of the model generation, such as loosening the objective bounds can help with this, or using the iterative approaches discussed later in this tutorial. 

All of these methods can be accessed through `metworkpy.imat.generate_model`:

In [ ]:
imat_restrictions_model = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="imat_restrictions",
    epsilon=1.0,
    threshold=1e-2,
)

In [ ]:
subset_model = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="subset",
    epsilon=1.0,
    threshold=1e-2,
)

In [ ]:
simple_model = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="simple",
    epsilon=1.0,
    threshold=1e-2,
)

In [ ]:
milp_model = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="milp",
    epsilon=1.0,
    threshold=1e-2,
)

In [ ]:
fva_model = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="fva",
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.0,
    loopless="fastSNP",
)

In [ ]:
# The resulting optimums for these models:
print(
    f"The maximum growth rate of the unconstrained model is: {model.slim_optimize():.4}"
)
print(
    f"The maximum growth of the the imat_restrictions model is: {imat_restrictions_model.slim_optimize():.4}"
)
print(
    f"The maximum growth of the the subset model is: {subset_model.slim_optimize():.4}"
)
print(
    f"The maximum growth of the the simple model is: {simple_model.slim_optimize():.4}"
)
print(
    f"The maximum growth of the the milp model is: {milp_model.slim_optimize():.4}"
)
print(
    f"The maximum growth of the the fva model is: {fva_model.slim_optimize():.4}"
)

## Sampling

With all of the methods other than the `imat_restrictions` the resulting models contain no integer constraints, and so can be used with COBRApy's flux sampling methods. For example, we can generate 
iMAT models for each of the 'gene weights' we generated earlier, and sample from them.

In [ ]:
imat_model1 = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights1,
    method="fva",
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.0,
)
imat_model2 = metworkpy.imat.generate_model(
    model,
    rxn_weights=reaction_weights2,
    method="fva",
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.0,
)

In [ ]:
imat_samples1 = cobra.sampling.sample(imat_model1, 1_000)
imat_samples2 = cobra.sampling.sample(imat_model2, 1_000)

Once we have samples from the iMAT models, there are a variety of ways to compare the flux distributions of the models, MetworkPy specifically includes divergence metrics in order to compare between models. 

In [ ]:
divergence = metworkpy.divergence.kl_divergence_functions.kl_divergence_array(
    imat_samples1,
    imat_samples2,
    n_neighbors=10,  # Number of neighbors to use for estimating divergence
    jitter=1e-10,  # Jitter shifts overlapping points so that they don't cause numerical problems
    jitter_seed=rng.integers(np.iinfo(np.int_).max),
    clip=True,
)

In [ ]:
divergence.to_frame("Divergence").sort_values(by="Divergence", ascending=False)

## Corner Sampling

In addition to sampling from the models generated from constraining the reactions in the model, MetworkPy includes an approach to sampling from the iMAT constrained model directly. This approach is corner sampling, which samples from the corners of the flux space. This is done by selecting a randomized subset of reactions, and using them to generate a random objective function, which when optimized yields a corner of the flux space. This is only garunteed to sample corners when the model doesn't include integer variables, but even with integer variables it does provide a sample of the edges of the flux space.

In [ ]:
imat_samples1 = metworkpy.imat.imat_sampling(
    model,
    rxn_weights=reaction_weights1,
    epsilon=1.0,
    threshold=1e-2,
    n_samples=500,
    seed=rng,
)
imat_samples2 = metworkpy.imat.imat_sampling(
    model,
    rxn_weights=reaction_weights2,
    epsilon=1.0,
    threshold=1e-2,
    n_samples=500,
    seed=rng,
)

In [ ]:
divergence = metworkpy.divergence.kl_divergence_functions.kl_divergence_array(
    imat_samples1,
    imat_samples2,
    permutation_estimation_method="empirical",
    n_neighbors=10,  # Number of neighbors to use for estimating divergence
    jitter=1e-10,  # Jitter shifts overlapping points so that they don't cause numerical problems
    jitter_seed=rng.integers(np.iinfo(np.int_).max),
    clip=True,
)

In [ ]:
divergence.to_frame("Divergence").sort_values(by="Divergence", ascending=False)

## Iterative iMAT

As mentioned above, the iMAT optimization problem can have multiple possible solutions that all achieve the same match between the gene expression data and the generated flux distribution. Above this was addressed by using FVA, or sampling from the model constrained with iMAT itself, but this can also be solved by iterating through the iMAT solutions in order to investigate the various different solutions. 

MetworkPy provides iterators for stepping through the iMAT solutions based on (Rodríguez-Mier et al. 2021) in several different ways. This includes different methods for determining the order of visiting the different iMAT solutions. The simplest method (called 'icut' for integer cut) is to add a constraint that stops the same solution from showing up again, and then optimizing the iMAT objective again. This garuntees a different solution, and eventually would step through all solutions, however it is generally infeasible to step through all of the solutions (since the number scales exponentially with the number of reactions with non-zero weights in iMAT), and the icut solutions tend to be close together. 

In order to address this, MetworkPy includes some methods for moving to more disimilar solutions (both of which still add the icut constraints to ensure the solutions don't repeat). The first is the 'maxdist' method from (Rodríguez-Mier et al. 2021), which maximizes the distance between successive solution. The second method is similar to the corner sampling discussed above, and uses a randomized objective function to try and explore the iMAT solutions.

MetworkPy includes different iterators for stepping through the solutions, including:
- ImatIterBinaryVariables: Returns a tuple of pandas Series describing which reactions are active and inactive, specifically the tuple is of:
  - rh_y_pos: The high expression reactions, with values of 1.0 for all the reactions which are active in the forward direction
  - rh_y_neg: The high expression reactions, with values of 1.0 for all the reactions which are active in the reverse direction
  - rl_y_pos: The low expression reactions, with values of 1.0 for all the reactions which are inactive
- ImatIterModels: Returns a Model constrained to follow the current iMAT solution, using either the 'subset' or 'simple' method described above.

In [ ]:
for idx, (rh_y_pos, rh_y_neg, rl_y_pos) in enumerate(
    metworkpy.imat.imat_iter.ImatIterBinaryVariables(
        model,
        rxn_weights=reaction_weights1,
        iter_method="corner",
        max_iter=20,
        epsilon=1.0,
        threshold=1e-2,
        objective_tolerance=0.1,
    )
):
    print(f"For model {idx}:")
    print(
        f"\tHigh Expression Active Forward: {list(rh_y_pos[rh_y_pos > 0.0].index)}"
    )
    print(
        f"\tHigh Expression Active Reverse: {list(rh_y_neg[rh_y_neg > 0.0].index)}"
    )
    print(
        f"\tLow Expression Inactive:        {list(rl_y_pos[rl_y_pos > 0.0].index)}"
    )

In [ ]:
for idx, imat_model_iter in enumerate(
    metworkpy.imat.imat_iter.ImatIterModels(
        model,
        rxn_weights=reaction_weights1,
        iter_method="corner",
        max_iter=20,
        epsilon=1.0,
        threshold=1e-2,
        objective_tolerance=0.1,
    )
):
    print(
        f"For model {idx} the maximum growth is: {imat_model_iter.slim_optimize()}"
    )

MetworkPy includes some wrappers around these iterators as well, for example to sample from each of the iMAT models:

In [ ]:
imat_samples1 = metworkpy.imat.imat_iter.imat_iter_flux_sample(
    model,
    rxn_weights=reaction_weights1,
    iter_method="corner",
    max_iter=20,
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.1,
    n_samples=100,
)
imat_samples2 = metworkpy.imat.imat_iter.imat_iter_flux_sample(
    model,
    rxn_weights=reaction_weights2,
    iter_method="corner",
    max_iter=20,
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.1,
    n_samples=100,
)
divergence = metworkpy.divergence.kl_divergence_functions.kl_divergence_array(
    imat_samples1,
    imat_samples2,
    permutation_estimation_method="empirical",
    n_neighbors=10,  # Number of neighbors to use for estimating divergence
    jitter=1e-10,  # Jitter shifts overlapping points so that they don't cause numerical problems
    jitter_seed=rng.integers(np.iinfo(np.int_).max),
    clip=True,
)
divergence.to_frame("Divergence").sort_values(by="Divergence", ascending=False)

MetworkPy also includes methods finding essential genes from each of the models:

In [ ]:
essential_genes_consensus = metworkpy.imat.imat_iter.imat_iter_essential(
    model,
    rxn_weights=reaction_weights1,
    essential_proportion=0.1,
    processes=1,
    iter_method="corner",
    max_iter=20,
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.1,
)
essential_genes_consensus

These essentiality predictions can be combined across the models to identify genes which are considered essential by a consensus of the iMAT models:

In [ ]:
metworkpy.imat.imat_iter.consensus_essentiality(
    model,
    rxn_weights=reaction_weights1,
    essential_proportion=0.1,
    consensus_method=0.8,
    processes=1,
    iter_method="corner",
    max_iter=20,
    epsilon=1.0,
    threshold=1e-2,
    objective_tolerance=0.1,
).to_frame("Essential?")

## References
- Rodríguez-Mier, P.; Poupin, N.; Blasio, C. de; Cam, L. L.; Jourdan, F. DEXOM: Diversity-Based Enumeration of Optimal Context-Specific Metabolic Networks. PLOS Computational Biology 2021, 17 (2), e1008730. https://doi.org/10.1371/journal.pcbi.1008730.
- Shlomi, T.; Cabili, M. N.; Herrgård, M. J.; Palsson, B. Ø.; Ruppin, E. Network-Based Prediction of Human Tissue-Specific Metabolism. Nat Biotechnol 2008, 26 (9), 1003–1010. https://doi.org/10.1038/nbt.1487.
- Zur, H.; Ruppin, E.; Shlomi, T. iMAT: An Integrative Metabolic Analysis Tool. Bioinformatics 2010, 26 (24), 3140–3142. https://doi.org/10.1093/bioinformatics/btq602.